# Bench reports — historical comparison

One section per `(benchmark, test_case)` pair across the most recent
`report_<N>.json` files written by the v2 bench framework. Each
section's table has:

* **columns** — every loaded report (oldest on the left, newest on the
  right);
* **rows** — every execution tuple (algorithm + parameter combination)
  that the bench evaluated for that `(benchmark, test_case)`;
* **values** — the median `wall_clock_ns` over the iterations that the
  bench framework recorded for that cell, formatted in human-friendly
  units (ns / µs / ms / s / min);
* **colour** — green when the cell is at least `TOLERANCE_PCT` faster
  than the cell to its left (smaller `wall_clock_ns`), red when it is
  at least `TOLERANCE_PCT` slower. The first column has no left
  neighbour and stays uncoloured.

Schema: matches the v2 bench framework's `report_<N>.json` layout —
`{ meta, benchmarks: { <name>: { execution_tuples: [{ tuple, test_cases: { <test_case>: { all_regions: { wall_clock_ns: [...] }}}}]}}}`.
The notebook walks every benchmark / tuple / test case it finds, so
new benchmarks added to the framework appear automatically on the
next notebook run.

In [1]:
# ----- imports + configuration -----
# REPORTING_ROOTS is every directory under main/reporting/ we scan for
# report_<N>.json files. Add a directory here when a new bench module
# starts writing reports; the discovery logic below treats every
# subdirectory equally.
#
# MAX_REPORTS_PER_BENCH caps how many reports we surface per bench
# (newest by index). Pick the most recent N reports per bench, then
# re-sort ascending so the table reads chronologically left-to-right.
#
# TOLERANCE_PCT is the symmetric window around "unchanged" — deltas
# inside [-TOLERANCE_PCT, +TOLERANCE_PCT] are noise; outside is a
# real speedup or regression. Tuned for the v2 bench's iteration
# counts (5–8 per cell on most matrices); raise it if you see noise
# colouring drown out signal on a quiet bench.

import json
import pathlib
import re
import statistics
from collections import defaultdict

import pandas as pd
from IPython.display import display, Markdown

REPORTING_ROOTS = [
    pathlib.Path('../reporting/bioseq_bench'),
    pathlib.Path('../reporting/bioseq_io_bench'),
    pathlib.Path('../reporting/bench_framework_example'),
]
MAX_REPORTS_PER_BENCH = 10
TOLERANCE_PCT = 5.0

REPORT_FILE_PATTERN = re.compile(r'^report_(\d+)\.json$')

print(f'TOLERANCE_PCT = {TOLERANCE_PCT}% (cells inside this window stay uncoloured)')
print(f'Scanning {len(REPORTING_ROOTS)} reporting root(s):')
for d in REPORTING_ROOTS:
    exists = d.exists()
    print(f'  {d.resolve()} {"[ok]" if exists else "[missing — will skip]"}')

TOLERANCE_PCT = 5.0% (cells inside this window stay uncoloured)
Scanning 3 reporting root(s):
  /Users/piotr.styczynski/Code/bioseqdb/main/reporting/bioseq_bench [ok]
  /Users/piotr.styczynski/Code/bioseqdb/main/reporting/bioseq_io_bench [ok]
  /Users/piotr.styczynski/Code/bioseqdb/main/reporting/bench_framework_example [ok]


In [2]:
# ----- discover + load reports per reporting root -----
# Walks every reporting root, picks the most-recent MAX_REPORTS_PER_BENCH
# `report_<N>.json` files (by integer suffix), and parses each into a
# dict carrying its index, source path, and parsed JSON. Reports that
# fail to parse are surfaced as a warning but don't stop the load —
# the bench writer occasionally lands a partial JSON when a run is
# killed mid-write, and we'd rather render the surviving reports than
# black-screen the notebook.
#
# Output: `reporting_buckets[bench_dir.name] = [report_obj, ...]` in
# chronological order (oldest first), where each `report_obj` has
# `{ index, path, commit, data }`.

def load_report_directory(root):
    if not root.exists():
        return []
    candidates = []
    for path in root.glob('report_*.json'):
        m = REPORT_FILE_PATTERN.match(path.name)
        if m:
            candidates.append((int(m.group(1)), path))
    if not candidates:
        return []
    candidates.sort(key=lambda t: t[0], reverse=True)
    candidates = candidates[:MAX_REPORTS_PER_BENCH]
    candidates.sort(key=lambda t: t[0])
    out = []
    for idx, path in candidates:
        try:
            with open(path, 'r') as fh:
                data = json.load(fh)
        except (json.JSONDecodeError, OSError) as exc:
            print(f'  skip {path.name}: {exc}')
            continue
        # Some bench runs land an empty `commit` (e.g. running in a
        # detached worktree where the harness couldn't read HEAD).
        # Normalise to the literal 'unknown' so column-header
        # truncation below doesn't render a blank column header.
        commit = data.get('meta', {}).get('commit') or 'unknown'
        out.append({
            'index':  idx,
            'path':   path,
            'commit': commit,
            'data':   data,
        })
    return out


reporting_buckets = {}
for root in REPORTING_ROOTS:
    reports = load_report_directory(root)
    if reports:
        reporting_buckets[root.name] = reports
        print(f'\n{root.name}: loaded {len(reports)} report(s)')
        for r in reports:
            short = (r['commit'][:12] + '…') if len(r['commit']) > 12 else r['commit']
            print(f'  report {r["index"]:>3}  commit {short:<14}  ({r["path"].name})')
    else:
        print(f'\n{root.name}: no reports')

if not reporting_buckets:
    raise SystemExit('No reports found under any of REPORTING_ROOTS.')


bioseq_bench: loaded 10 report(s)
  report   3  commit e4aa89cec21e…   (report_3.json)
  report   4  commit 5f65bf04e4da…   (report_4.json)
  report   5  commit 5f65bf04e4da…   (report_5.json)
  report   6  commit 5f65bf04e4da…   (report_6.json)
  report   7  commit 5f65bf04e4da…   (report_7.json)
  report   8  commit 8515703caee0…   (report_8.json)
  report   9  commit 8515703caee0…   (report_9.json)
  report  10  commit 8515703caee0…   (report_10.json)
  report  11  commit 8515703caee0…   (report_11.json)
  report  12  commit 8515703caee0…   (report_12.json)

bioseq_io_bench: loaded 6 report(s)
  report   1  commit e4aa89cec21e…   (report_1.json)
  report   2  commit e4aa89cec21e…   (report_2.json)
  report   3  commit e4aa89cec21e…   (report_3.json)
  report   4  commit 00dda45bbf8b…   (report_4.json)
  report   5  commit 00dda45bbf8b…   (report_5.json)
  report   6  commit 00dda45bbf8b…   (report_6.json)

bench_framework_example: loaded 10 report(s)
  report   1  commit b46b5dadea

In [3]:
# ----- helpers: tuple labelling, value extraction, time formatting -----
# `tuple_label` flattens the v2 framework's `tuple` dict
#   { 'algo_name': { 'param1': v1, 'param2': v2, ... } }
# into a single human-readable string. The framework wraps the
# algorithm key around its parameters; flattening makes the row label
# easy to read even when an execution tuple is fully parameterised.
#
# `wall_clock_median_ns` reduces a sample list to a single representative
# point. We use the median rather than the mean because the bench's
# regression rules pin p50, and the mean is sensitive to a single cold-
# cache outlier on the small synthetic cells. Returns None when there
# is no measurement (the cell wasn't run, or the bench skipped it).
#
# `format_time_ns` picks the most readable unit for the reduced value
# (ns / µs / ms / s / min). Tabular layout is preserved across rows
# because the column width is set by the widest formatted string.

def tuple_label(tuple_dict):
    if not isinstance(tuple_dict, dict) or not tuple_dict:
        return '<unknown>'
    parts = []
    for algo, params in tuple_dict.items():
        if isinstance(params, dict) and params:
            # Sort params for determinism — the JSON order of dict
            # keys is implementation-defined under JSON-Schema, even
            # though Python's dict preserves insertion order.
            kv = ', '.join(f'{k}={v}' for k, v in sorted(params.items())
                           if k != '_')
            parts.append(f'{algo}({kv})' if kv else algo)
        else:
            parts.append(str(algo))
    return ' / '.join(parts)


def wall_clock_median_ns(test_case_block):
    if not isinstance(test_case_block, dict):
        return None
    samples = test_case_block.get('all_regions', {}).get('wall_clock_ns', [])
    if not samples:
        return None
    try:
        return float(statistics.median(samples))
    except (TypeError, ValueError, statistics.StatisticsError):
        return None


def format_time_ns(value_ns):
    '''Render `value_ns` (nanoseconds) in the most readable unit.
    Uses the SI / time-of-day breakpoints common in profiler output.
    Returns the literal '\u2014' (em dash) for None so missing cells
    visually contrast against measured ones.'''
    if value_ns is None:
        return '\u2014'
    if value_ns < 1_000:
        return f'{value_ns:.0f} ns'
    if value_ns < 1_000_000:
        return f'{value_ns / 1_000:.2f} \u00b5s'
    if value_ns < 1_000_000_000:
        return f'{value_ns / 1_000_000:.2f} ms'
    if value_ns < 60_000_000_000:
        return f'{value_ns / 1_000_000_000:.2f} s'
    return f'{value_ns / 60_000_000_000:.2f} min'

In [4]:
# ----- table construction + colouring -----
# `enumerate_benchmark_test_cases` walks the loaded reports and yields
# every `(reporting_dir, benchmark_name, test_case_name)` triple that
# appears in at least one report. We yield in a stable order so that
# re-running the notebook produces the same section sequence:
#   * outer key: reporting directory name (sorted)
#   * middle:    benchmark name (first-appearance order across reports)
#   * inner:     test case name (first-appearance order across reports)
#
# `build_test_case_dataframe` produces the table for one
# (benchmark, test_case) pair: rows are execution tuples that ran on
# the test case in any of the loaded reports (first-appearance order;
# tuples disappearing from a newer report show up with NaN cells in
# the newer columns); columns are reports in chronological order with
# the report number as the visible header.
#
# `colour_history` shades each cell relative to its left neighbour
# (the previous report's value for the same row). For wall-clock the
# sense is lower-is-better, so a cell that grew vs. its neighbour is
# RED, and a cell that shrunk is GREEN. Cells inside the tolerance
# window stay uncoloured. Colour intensity scales with how far past
# tolerance the delta is, capped at 20 percentage points beyond
# tolerance for readability.

def enumerate_benchmark_test_cases(reporting_buckets):
    for reporting_name in sorted(reporting_buckets):
        reports = reporting_buckets[reporting_name]
        # Bench-name + test-case ordering: first appearance across reports.
        bench_order = []
        bench_seen = set()
        bench_to_test_cases = defaultdict(list)
        bench_test_case_seen = defaultdict(set)
        for r in reports:
            for bname, bench in r['data'].get('benchmarks', {}).items():
                if bname not in bench_seen:
                    bench_order.append(bname)
                    bench_seen.add(bname)
                seen_tcs = bench_test_case_seen[bname]
                for tup in bench.get('execution_tuples', []):
                    for tc_name in tup.get('test_cases', {}):
                        if tc_name not in seen_tcs:
                            seen_tcs.add(tc_name)
                            bench_to_test_cases[bname].append(tc_name)
        for bname in bench_order:
            for tc_name in bench_to_test_cases[bname]:
                yield (reporting_name, bname, tc_name)


def build_test_case_dataframe(reporting_name, bench_name, tc_name):
    reports = reporting_buckets[reporting_name]
    columns = pd.MultiIndex.from_tuples(
        [(r['commit'][:12] if r['commit'] != 'unknown' else 'unknown',
          f'report {r["index"]}') for r in reports],
        names=['Commit', 'Report'],
    )
    # Discover tuples in first-appearance order across reports.
    tuple_order = []
    tuple_seen = set()
    for r in reports:
        bench = r['data'].get('benchmarks', {}).get(bench_name, {})
        for tup in bench.get('execution_tuples', []):
            label = tuple_label(tup.get('tuple', {}))
            if tc_name in tup.get('test_cases', {}) and label not in tuple_seen:
                tuple_seen.add(label)
                tuple_order.append(label)
    if not tuple_order:
        return None
    # Pivot into rows × columns.
    rows = []
    for label in tuple_order:
        row = []
        for r in reports:
            bench = r['data'].get('benchmarks', {}).get(bench_name, {})
            value = None
            for tup in bench.get('execution_tuples', []):
                if tuple_label(tup.get('tuple', {})) != label:
                    continue
                value = wall_clock_median_ns(
                    tup.get('test_cases', {}).get(tc_name))
                break
            row.append(value)
        rows.append(row)
    return pd.DataFrame(rows, index=tuple_order, columns=columns)


def colour_history(df, tolerance_pct):
    '''Per-row neighbour comparison. Wall-clock semantics: lower is
    better, so growth (delta > tolerance) is RED and shrinkage
    (delta < -tolerance) is GREEN. The first column and any cell whose
    left neighbour is missing stay uncoloured.
    '''
    def row_styles(values):
        styles = ['' for _ in values]
        prev = None
        for i, value in enumerate(values):
            if pd.isna(value):
                continue
            if prev is not None and prev > 0:
                delta_pct = (value - prev) / prev * 100.0
                if delta_pct < -tolerance_pct:
                    # faster → green
                    intensity = min(1.0, (abs(delta_pct) - tolerance_pct) / 20.0)
                    rb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb({rb}, 255, {rb}); '
                                 f'color: black; font-weight: 600;')
                elif delta_pct > tolerance_pct:
                    # slower → red
                    intensity = min(1.0, (delta_pct - tolerance_pct) / 20.0)
                    gb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb(255, {gb}, {gb}); '
                                 f'color: black; font-weight: 600;')
            prev = value
        return styles
    return (df.style
              .apply(row_styles, axis=1)
              .format(format_time_ns, na_rep='\u2014')
              .set_table_styles([
                  {'selector': 'th',
                   'props': 'text-align: center; padding: 4px 10px;'},
                  {'selector': 'td',
                   'props': 'text-align: right; padding: 4px 10px; '
                            'font-variant-numeric: tabular-nums;'},
              ]))

## Per-test-case wall-clock history

One section per `(benchmark, test_case)` pair. Each section's heading
encodes both:

* the **benchmark name** (e.g. `bench_io_writers_fasta`) — the C++
  unit declared in the bench's JSON config (one per `matrix` entry);
* the **test case name** (e.g. `FASTA_SHORT`, `FASTQ_REAL_HUMAN_FASTQ_GZ`)
  — the corpus the bench iterates against under that benchmark.

Same `(benchmark, test_case)` heading might appear in two reporting
directories if both benches measure it — a deliberate duplication
(separate evaluation axes, separate regression baselines). The
directory is shown above the heading.

In [5]:
# ----- render: one section per (reporting_dir, benchmark, test_case) -----
# Walks the discovered triples in deterministic order, builds a
# DataFrame for each, and renders a markdown heading + the styled
# table. Headings carry both the benchmark name and the test-case
# label so a reader can locate the corresponding bench config block
# without grepping. Sections that have no rows (empty `execution_tuples`,
# or every tuple missing this test case) are silently skipped — the
# bench framework occasionally records empty matrices when a config
# block has no `for` cells.

current_reporting_dir = None
current_benchmark     = None
rendered_section_count = 0

for reporting_name, bench_name, tc_name in enumerate_benchmark_test_cases(
        reporting_buckets):
    df = build_test_case_dataframe(reporting_name, bench_name, tc_name)
    if df is None or df.empty:
        continue
    # Top-level heading: reporting directory (only on change). Helps when
    # a single notebook compares the aligner bench and the IO bench in
    # one go — the reporting-dir heading delineates the two domains.
    if reporting_name != current_reporting_dir:
        display(Markdown(f'# Reporting directory: `{reporting_name}`'))
        current_reporting_dir = reporting_name
        current_benchmark = None
    # Mid heading: benchmark name (only on change).
    if bench_name != current_benchmark:
        display(Markdown(f'## Benchmark: `{bench_name}`'))
        current_benchmark = bench_name
    # Per-test-case heading + table.
    display(Markdown(f'### Test case: `{tc_name}`'))
    display(colour_history(df, TOLERANCE_PCT))
    rendered_section_count += 1

if rendered_section_count == 0:
    display(Markdown('*No (benchmark, test_case) pairs had data — '
                     'check that REPORTING_ROOTS contains report_<N>.json '
                     'files written by the v2 bench framework.*'))
else:
    print(f'\nRendered {rendered_section_count} (benchmark, test_case) section(s).')

# Reporting directory: `bench_framework_example`

## Benchmark: `bench_count`

### Test case: `TINY`

### Test case: `SMALL`

## Benchmark: `bench_sort`

...

### Test case: `FASTQ_MEDIUM`

### Test case: `FASTQ_LONG`

### Test case: `FASTQ_VERY_LONG`

### Test case: `FASTQ_REAL_HUMAN_FASTQ_6GB`


Rendered 51 (benchmark, test_case) section(s).


## Full commit hashes

Per-reporting-directory list of every commit hash referenced in the
column headers above. The headers truncate to the first 12 characters
for column-width reasons; if a green / red cell points at a specific
change, copy the full hash from here for `git show <hash>`.

In [6]:
# Per reporting dir, list (report N, full commit hash). Sorted by
# report number ascending — same order as the table columns above.
for reporting_name in sorted(reporting_buckets):
    reports = reporting_buckets[reporting_name]
    lines = [f'### `{reporting_name}`', '']
    for r in reports:
        dirty = ' *(dirty)*' if r['data'].get('meta', {}).get('dirty') else ''
        lines.append(f'* **report {r["index"]}** \u2014 `{r["commit"]}`{dirty}')
    display(Markdown('\n'.join(lines)))

### `bench_framework_example`

* **report 1** — `b46b5dadea5f89d532ac95703fbe347c8426dd1c` *(dirty)*
* **report 2** — `unknown`
* **report 3** — `unknown`
* **report 4** — `unknown`
* **report 5** — `unknown`
* **report 6** — `04c357ac07905f9412b3cc989225b572f663ae69` *(dirty)*
* **report 7** — `04c357ac07905f9412b3cc989225b572f663ae69` *(dirty)*
* **report 8** — `bda813e1dd5847f67077086b35ad339b4e0347d4` *(dirty)*
* **report 9** — `bda813e1dd5847f67077086b35ad339b4e0347d4` *(dirty)*
* **report 10** — `unknown`

### `bioseq_bench`

* **report 3** — `e4aa89cec21e53cff4999bf790a4f9484a7847fa` *(dirty)*
* **report 4** — `5f65bf04e4da1e06e5849cc965486626609c21f2` *(dirty)*
* **report 5** — `5f65bf04e4da1e06e5849cc965486626609c21f2` *(dirty)*
* **report 6** — `5f65bf04e4da1e06e5849cc965486626609c21f2` *(dirty)*
* **report 7** — `5f65bf04e4da1e06e5849cc965486626609c21f2` *(dirty)*
* **report 8** — `8515703caee01155701ca7099296bb7f9cdded83` *(dirty)*
* **report 9** — `8515703caee01155701ca7099296bb7f9cdded83` *(dirty)*
* **report 10** — `8515703caee01155701ca7099296bb7f9cdded83` *(dirty)*
* **report 11** — `8515703caee01155701ca7099296bb7f9cdded83` *(dirty)*
* **report 12** — `8515703caee01155701ca7099296bb7f9cdded83` *(dirty)*

### `bioseq_io_bench`

* **report 1** — `e4aa89cec21e53cff4999bf790a4f9484a7847fa` *(dirty)*
* **report 2** — `e4aa89cec21e53cff4999bf790a4f9484a7847fa` *(dirty)*
* **report 3** — `e4aa89cec21e53cff4999bf790a4f9484a7847fa` *(dirty)*
* **report 4** — `00dda45bbf8bcc1e44ad42db1e007cada53f004c` *(dirty)*
* **report 5** — `00dda45bbf8bcc1e44ad42db1e007cada53f004c` *(dirty)*
* **report 6** — `00dda45bbf8bcc1e44ad42db1e007cada53f004c` *(dirty)*